# PV7: pemulihan VRAM setelah GPU OOM (AUDIT R12)

Membuktikan di GPU nyata bahwa layanan inference pulih sendiri setelah GPU kehabisan memori:
OOM menjadi error berkode `gpu_oom` (bukan crash), lock model dilepas, dan frame berikutnya
diklasifikasikan normal dengan **teks model yang sama persis** seperti sebelum OOM.

Jalur yang diuji adalah kode layanan sendiri (`app.pipeline.classify_bytes` + `GuardedBackend` +
`RealBackend`), tanpa server HTTP. Pemetaan `gpu_oom` → HTTP 503 sudah diuji dengan backend palsu.

1. **Baseline**: 2 inferensi normal (teks, waktu, memori).
2. **Uji A**: alokasi 64 GB di luar model → OOM, lalu inferensi normal lagi.
3. **Uji B**: VRAM ditahan sampai hampir habis, lalu `classify_bytes` → OOM di dalam `generate`.
   Cek: `BackendFailure("gpu_oom", 503)`, kedua lock terlepas. Penahan dilepas **tanpa**
   `empty_cache` tambahan (layanan sendiri sudah memanggilnya di cabang OOM), lalu inferensi normal.
4. Memori sebelum dan sesudah `torch.cuda.empty_cache()`.

**Sebelum Run all:** Runtime → Change runtime type → **T4 GPU**. Unggah notebook ini lewat
File → Upload notebook. Model diunduh dari Hugging Face (±5–10 menit pertama kali).

Hasil: `pv7_results.zip` (unduh otomatis di sel terakhir). Kirim file itu kembali. Setiap uji mencatat
error-nya sendiri, jadi satu uji yang gagal tidak menghentikan uji berikutnya.

In [ ]:
REPO = "https://github.com/WANGDELL30/agronova-poultry-dashboard"
BRANCH = "fix/web-batch-1"
IMAGE = "data/working/dataset/test/stress/stress_0001.jpg"   # crop berlabel di repo
FREE_TARGETS_MB = [64, 16, 4]   # Uji B: sisa VRAM yang dicoba sampai generate kehabisan memori

In [ ]:
!rm -rf /content/repo && git clone -q -b $BRANCH $REPO /content/repo
%cd /content/repo/apps/inference
!pip -q install -r requirements.txt -r requirements-real.txt
# Colab membawa torchao lama yang bentrok dengan transformers baru; tidak dipakai di sini
!pip -q uninstall -y torchao

In [ ]:
import json, os, secrets, subprocess, sys, threading, time, traceback
from pathlib import Path

import torch

sys.path.insert(0, "/content/repo/apps/inference")
os.environ["INFERENCE_MODE"] = "real"
os.environ.setdefault("INFERENCE_API_KEY", secrets.token_urlsafe(16))  # wajib di mode real; tidak dipakai
from app.backends import RealBackend
from app.config import load_settings
from app.pipeline import BackendFailure, GuardedBackend, classify_bytes

MB = 1024 ** 2
results = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "errors": {}}


def mem(label):
    free, total = torch.cuda.mem_get_info()
    row = {"label": label, "allocated_mb": round(torch.cuda.memory_allocated() / MB),
           "reserved_mb": round(torch.cuda.memory_reserved() / MB), "free_mb": round(free / MB),
           "total_mb": round(total / MB)}
    results.setdefault("memory", []).append(row)
    print(row)
    return row


t0 = time.perf_counter()
real = RealBackend(load_settings())
results["load_s"] = round(time.perf_counter() - t0, 1)
model_lock = threading.Lock()   # sama seperti State.model_lock di app.main
backend = GuardedBackend(real, model_lock, lambda: False)
image = (Path("/content/repo") / IMAGE).read_bytes()
mem("setelah load")


def run(label):
    r = classify_bytes(image, backend)
    row = {"label": label, "status": r["status"], "label_pred": (r["result"] or {}).get("label"),
           "inference_ms": r["inference_ms"], "raw": r["raw_response"]}
    results.setdefault("runs", []).append(row)
    print(label, row["status"], row["label_pred"], row["inference_ms"], "ms")
    return row

In [ ]:
# 1. Baseline
base1 = run("baseline 1")
base2 = run("baseline 2")
results["baseline_deterministic"] = base1["raw"] == base2["raw"]
mem("setelah baseline")

In [ ]:
# 2. Uji A: alokasi raksasa di luar model
a = {}
try:
    try:
        torch.empty(64 * 1024 ** 3, dtype=torch.uint8, device="cuda")
        a["oom_raised"] = False
    except torch.cuda.OutOfMemoryError as exc:
        a["oom_raised"] = True
        a["error_type"] = type(exc).__name__
    mem("A: setelah OOM 64 GB")
    after = run("A: setelah OOM")
    a["same_text_as_baseline"] = after["raw"] == base1["raw"]
    a["status"] = after["status"]
except Exception:
    results["errors"]["A"] = traceback.format_exc()
    print(results["errors"]["A"])
results["test_a"] = a
print(a)

In [ ]:
# 3. Uji B: OOM di dalam generate
b = {"attempts": []}
holder = []
try:
    for target in FREE_TARGETS_MB:
        # Isi VRAM sampai sisa sekitar `target` MB (blok besar dulu, lalu makin kecil).
        for block_mb in (1024, 256, 64, 16, 4, 1):
            while torch.cuda.mem_get_info()[0] > (target + block_mb) * MB:
                try:
                    holder.append(torch.empty(block_mb * MB, dtype=torch.uint8, device="cuda"))
                except torch.cuda.OutOfMemoryError:
                    break
        free_before = round(torch.cuda.mem_get_info()[0] / MB)
        attempt = {"target_free_mb": target, "free_before_mb": free_before}
        try:
            r = classify_bytes(image, backend)
            attempt["outcome"] = "no_oom"
            attempt["status"] = r["status"]
        except BackendFailure as failure:
            attempt["outcome"] = "backend_failure"
            attempt["code"], attempt["http_status"] = failure.code, failure.http_status
        except Exception as exc:  # noqa: BLE001 - recorded, the point is to see what happened
            attempt["outcome"] = f"unexpected {type(exc).__name__}: {exc}"[:300]
        attempt["model_lock_released"] = not model_lock.locked()
        attempt["backend_lock_released"] = not real._lock.locked()
        b["attempts"].append(attempt)
        print(attempt)
        if attempt["outcome"] != "no_oom":
            break
    mem("B: masih ditahan")
    # Lepaskan penahan tanpa empty_cache tambahan: layanan sendiri sudah memanggilnya di cabang OOM.
    holder.clear()
    mem("B: penahan dilepas, tanpa empty_cache")
    recovered = run("B: setelah OOM")
    b["recovered_status"] = recovered["status"]
    b["same_text_as_baseline"] = recovered["raw"] == base1["raw"]
    b["recovered_ms"] = recovered["inference_ms"]
    b["baseline_ms"] = base2["inference_ms"]
except Exception:
    results["errors"]["B"] = traceback.format_exc()
    print(results["errors"]["B"])
finally:
    holder.clear()
results["test_b"] = b

In [ ]:
# 4. Memori sebelum/sesudah empty_cache, lalu satu inferensi terakhir
mem("sebelum empty_cache")
torch.cuda.empty_cache()
mem("sesudah empty_cache")
final = run("akhir")
results["final_same_text"] = final["raw"] == base1["raw"]
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
import zipfile
from google.colab import files

b = results.get("test_b", {})
last = (b.get("attempts") or [{}])[-1]
results["verdict"] = {
    "A_oom_then_same_text": bool(results.get("test_a", {}).get("oom_raised") and results["test_a"].get("same_text_as_baseline")),
    "B_gpu_oom_code": last.get("code") == "gpu_oom" and last.get("http_status") == 503,
    "B_locks_released": bool(last.get("model_lock_released") and last.get("backend_lock_released")),
    "B_recovered_same_text": b.get("recovered_status") == "classified" and bool(b.get("same_text_as_baseline")),
    "baseline_deterministic": results.get("baseline_deterministic"),
    "no_errors": not results["errors"],
}
for run_row in results.get("runs", []):
    run_row["raw"] = (run_row["raw"] or "")[:2000]
Path("/content/pv7_results.json").write_text(json.dumps(results, indent=1, default=str))
with zipfile.ZipFile("/content/pv7_results.zip", "w") as z:
    z.write("/content/pv7_results.json", "pv7_results.json")
print(json.dumps(results["verdict"], indent=1))
files.download("/content/pv7_results.zip")